<a href="https://colab.research.google.com/github/AnithaN21/AI-12-days-Bootcamp/blob/main/LAB_9A_AGENT_CREATED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search

import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Gemini API key: ··········


In [2]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 907.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 2.3 MB/s eta 0:00:00


In [3]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

/tmp/ipykernel_6775/752164138.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


24 Apr 2026 TCS deepens partnership with Google Cloud to power AI-native autonomous enterprises 23 Mar 2026 TCS study reveals higher education at a digital crossroads: 61% of universities lag in digital maturity 03 Mar 2026 Fortune Names TCS to World's Most Admired Companies™ List for Fourth Straight Year Discover the latest TCS hiring 2026 updates, including TCS recruitment 2026, upcoming TCS job


In [4]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_6775/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [5]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

# Print every message in the conversation
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': '05c82de3-0d55-402d-8973-98972ebe3b4e', 'type': 'tool_call'}]

[2] ToolMessage
    Content: 2026年2月24日 · TCS has opened Engineering Fresher Hiring through the National Qualifier Test (NQT) for Digital & Prime roles. Eligible batches: 2024 | 2025 | 2026. Degrees: ... 2026年3月12日 · Rank-based counselling, examination guidance aur management quota admission ke liye turant call karein. ... TOP 

[3] AIMessage
    Content: [{'type': 'text', 'text': 'I couldn\'t find a specific "hiring quota" for TCS for the year 2026 in my search results. The information available pertains to their NQT (National Qualifier Test) for batches up to 2026 and some general hiring trends for the IT sector and TCS for the fiscal year 2025-26,


## Day 9 Lab 9A — Trace as a story

1. **Human asked:** "What is TCS's 2026 hiring quota?"

2. **Agent thought:** "I need recent information, so I should use the web_search tool."

3. **Agent acted:** called `web_search('TCS 2026 hiring quota')`.

4. **Agent observed:** search results related to TCS NQT hiring and 2026 batches.

5. **Agent answered:** it could not find an exact hiring quota but summarized the available hiring information.

This demonstrates the ReAct loop:
Human → Thought → Tool Call → Observation → Final Answer

In [6]:
# Pass a bad URL to trigger failure
result = agent.invoke({
    'messages': [('user',
                  'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

# Print the full trace
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')

    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')


[0] HumanMessage
    Content: Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    Content: [{'type': 'text', 'text': 'I cannot directly access the content of a URL. My `web_search` tool can search the web for information using a query, but it cannot open a specific URL and tell you what it says.', 'extras': {'signature': 'CukFAQw51scalSamTovtY9qe2OfSdHlmCyN7mGqbo676ORXbRkpP9vy1J4FZVunwU9I


## Step 6 — Failure Recovery Observation

The agent received a request to search a non-existent URL.

The agent understood that the `web_search` tool cannot directly open URLs and explained the limitation clearly instead of generating fake information.

This shows good failure handling and demonstrates responsible agent behaviour.

Observed behaviour:
- No hallucination
- Clear explanation of tool limitation
- Safe response to invalid input